# IOAI — 2025 Selection 2 Dieting Network (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-selection-2-dieting-network'
for f in ['X_train.pt','y_train.pt','X_val.pt']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Dieting Network — MAIO 2025 (Selection 2)

이 회귀 신경망은 **모든 가중치 = 1, 모든 바이어스 = 0** 으로 얼어붙어 있다(학습 불가). 바꿀 수 있는 것은
오직 **활성함수(`nn.Identity()` 자리)** 뿐이다. 어떤 활성함수를 넣어야 이 망이 제 역할을 할까?

`data/X_train.pt`, `data/y_train.pt` 로 목표를 파악하고, `data/X_val.pt` 를 예측해 `y_val_pred.pt` 를 저장하면
검증셋 정답과의 **R²** 로 채점한다(정답 라벨은 숨김).


In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import r2_score

X_train = torch.load("data/X_train.pt", weights_only=False)
y_train = torch.load("data/y_train.pt", weights_only=False)
X_val   = torch.load("data/X_val.pt",   weights_only=False)
print(X_train.shape, y_train.shape, X_val.shape)

In [ ]:
class FixedLinear(nn.Module):
    """nn.Linear 과 비슷하나 학습 불가 — weight 는 전부 1, bias 없음."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = torch.ones(out_features, in_features)
    def forward(self, x):
        return torch.mm(x, self.weight.t())

class DietNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = FixedLinear(8, 5)
        self.act1 = nn.Identity()   # <- Replace me!
        self.layer2 = FixedLinear(5, 5)
        self.act2 = nn.Identity()   # <- Replace me!
        self.layer3 = FixedLinear(5, 5)
        self.act3 = nn.Identity()   # <- Replace me!
        self.layer4 = FixedLinear(5, 1)
    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.act2(self.layer2(x))
        x = self.act3(self.layer3(x))
        return self.layer4(x)

In [ ]:
model = DietNetwork().eval()
with torch.no_grad():
    print("train R2:", round(r2_score(y_train, model(X_train)), 4))

In [ ]:
with torch.no_grad():
    y_val_pred = model(X_val)
torch.save(y_val_pred, "y_val_pred.pt")
print("saved y_val_pred.pt", tuple(y_val_pred.shape))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['y_val_pred.pt']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)